# KernelRidge RBF

Kernel Ridge combines ridge regression with an RBF kernel. It is useful as another nonlinear baseline.

## Shared Setup

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.impute import SimpleImputer
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# Colab data setup: upload train.csv and test.csv if the data folder is missing.
DATA_DIR = Path("data")
required_files = ["train.csv", "test.csv"]

if not all((DATA_DIR / name).exists() for name in required_files):
    DATA_DIR.mkdir(exist_ok=True)
    try:
        from google.colab import files
        print("Upload train.csv and test.csv. You may also upload sample_submission.csv and data_description.txt.")
        uploaded = files.upload()
        for filename, content in uploaded.items():
            (DATA_DIR / filename).write_bytes(content)
    except ModuleNotFoundError:
        print("Not running in Colab. Put train.csv and test.csv inside a local data/ folder.")

missing = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing required data files in {DATA_DIR}: {missing}")

print("Data folder ready:", sorted(path.name for path in DATA_DIR.glob("*")))


## Load, Clean, Feature Engineer, and Build Pipelines

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

absent_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "MasVnrType",
    "FireplaceQu",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
]

for col in absent_cols:
    if col in train.columns:
        train[col] = train[col].fillna("None")
    if col in test.columns:
        test[col] = test[col].fillna("None")

for df in (train, test):
    if "GarageYrBlt" in df.columns:
        df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)

outlier_mask = (train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)
train = train.loc[~outlier_mask].copy()

def add_features(df):
    df = df.copy()
    for col in ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath"]:
        if col not in df.columns:
            df[col] = 0
    df["TotalSF"] = df["TotalBsmtSF"].fillna(0) + df["1stFlrSF"].fillna(0) + df["2ndFlrSF"].fillna(0)
    df["TotalBath"] = (
        df["FullBath"].fillna(0)
        + 0.5 * df["HalfBath"].fillna(0)
        + df["BsmtFullBath"].fillna(0)
        + 0.5 * df["BsmtHalfBath"].fillna(0)
    )
    if {"YrSold", "YearBuilt"}.issubset(df.columns):
        df["HouseAge"] = (df["YrSold"] - df["YearBuilt"]).clip(lower=0)
    if {"YrSold", "YearRemodAdd"}.issubset(df.columns):
        df["YearsSinceRemodel"] = (df["YrSold"] - df["YearRemodAdd"]).clip(lower=0)
    if {"OverallQual", "GrLivArea"}.issubset(df.columns):
        df["QualityArea"] = df["OverallQual"] * df["GrLivArea"]
    return df

train = add_features(train)
test = add_features(test)

X = train.drop(columns=["Id", "SalePrice"], errors="ignore")
y = np.log(train["SalePrice"])
test_X = test.drop(columns=["Id"], errors="ignore")

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include="object").columns.tolist()

try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

linear_preprocessor = ColumnTransformer(
    [
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", onehot)]), categorical_features),
    ]
)

tree_preprocessor = ColumnTransformer(
    [
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", onehot)]), categorical_features),
    ]
)

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def rmsle_cv(model):
    scores = np.sqrt(-cross_val_score(model, X, y, cv=cv, scoring="neg_mean_squared_error", n_jobs=1))
    return scores.mean(), scores.std()

def save_submission(model, name):
    log_pred = model.predict(test_X)
    submission = pd.DataFrame({"Id": test["Id"], "SalePrice": np.exp(log_pred).clip(min=1)})
    path = OUTPUT_DIR / f"submission_{name}.csv"
    submission.to_csv(path, index=False)
    return path, submission.head()

## Exploratory Data Analysis

        This EDA section is included before model training so each model notebook is self-contained. It checks the target distribution, missing values, strongest numerical correlations, outliers, and the full modeling workflow.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(train["SalePrice"], kde=True, ax=axes[0], color="#4169e1")
axes[0].set_title(f"Raw SalePrice | skew = {train['SalePrice'].skew():.2f}")
axes[0].set_xlabel("SalePrice")

sns.histplot(np.log(train["SalePrice"]), kde=True, ax=axes[1], color="#2e8b57")
axes[1].set_title(f"log(SalePrice) | skew = {np.log(train['SalePrice']).skew():.2f}")
axes[1].set_xlabel("log(SalePrice)")

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False).head(20)
missing_table = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": (missing / len(train) * 100).round(2),
})

display(missing_table)

if not missing_table.empty:
    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=missing_table.reset_index().rename(columns={"index": "feature"}),
        y="feature",
        x="missing_percent",
        color="#d95f02",
    )
    plt.title("Top Missing-Value Features")
    plt.xlabel("Missing values (%)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()
plt.close()

In [ ]:
numeric_for_corr = train.select_dtypes(include=np.number).columns
corr_target = train[numeric_for_corr].corr(numeric_only=True)["SalePrice"].abs().sort_values(ascending=False)
display(corr_target.head(15).to_frame("abs_corr_with_saleprice"))

top_features = corr_target.head(11).index
plt.figure(figsize=(10, 8))
sns.heatmap(
    train[top_features].corr(numeric_only=True),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
)
plt.title("Correlation Heatmap: Top SalePrice Drivers")
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=train, x="GrLivArea", y="SalePrice", hue="OverallQual", palette="viridis", legend=False)
plt.title("Outlier Check: Living Area vs Sale Price")
plt.xlabel("Above-ground living area")
plt.ylabel("SalePrice")
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis("off")

steps = [
    "Load data",
    "Clean missing values",
    "Feature engineering",
    "Preprocess",
    "Train model",
    "CV RMSLE",
    "Submission CSV",
]
x_positions = np.linspace(0.06, 0.94, len(steps))

for idx, (x_pos, label) in enumerate(zip(x_positions, steps)):
    ax.text(
        x_pos,
        0.55,
        label,
        ha="center",
        va="center",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#f5f7fb", edgecolor="#4c566a"),
        transform=ax.transAxes,
    )
    if idx < len(steps) - 1:
        ax.annotate(
            "",
            xy=(x_positions[idx + 1] - 0.055, 0.55),
            xytext=(x_pos + 0.055, 0.55),
            arrowprops=dict(arrowstyle="->", color="#4c566a", lw=1.5),
            xycoords=ax.transAxes,
            textcoords=ax.transAxes,
        )

ax.set_title("End-to-End Modeling Workflow", fontsize=13, pad=16)
plt.show()
plt.close()

## Train, Evaluate, and Generate Submission

In [ ]:
model = Pipeline([
    ("preprocessor", linear_preprocessor),
    ("model", KernelRidge(kernel="rbf", alpha=0.1, gamma=0.001)),
])
mean, std = rmsle_cv(model)
model.fit(X, y)
print(f"KernelRidge RBF CV RMSLE = {mean:.5f} +/- {std:.5f}")
save_submission(model, "kernelridge_rbf")